# Comportement bovin — R(2+1)D-18 sur CVB (Colab GPU)

**Cache intelligent** : la 1ʳᵉ fois on décompresse + on construit un petit
cache (~200 Mo). Ensuite, chaque session recharge le cache en **secondes**,
**sans re-décompresser**. Runtime → GPU avant de lancer.


## 1. GPU


In [ ]:
import torch
print('CUDA :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(active le GPU !)')

## 2. Drive + emplacement du zip / cache


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ton zip CVB dans Drive (raccourci si partagé — cf. options plus bas)
ZIP_PATH = '/content/drive/MyDrive/cvb.zip'          # <<< AJUSTE
# Le cache est écrit ICI (créé 1 fois, réutilisé ensuite)
CACHE    = '/content/drive/MyDrive/cvb_clips_cache.npz'

# --- si le zip est PARTAGÉ par lien, décommente : ---
# !pip -q install -U gdown
# import gdown; ZIP_PATH='/content/cvb.zip'
# gdown.download(id='TON_ID', output=ZIP_PATH, quiet=False)

## 3. Clips — cache une fois (5 classes fusionnées)

mange · marche · debout · couché · boit


In [ ]:
import re, os, glob, cv2, numpy as np, zipfile
from collections import Counter

BEH_MAP={2:'grazing',3:'walking',4:'ruminating-standing',5:'ruminating-lying',
         6:'resting-standing',7:'resting-lying',8:'drinking',12:'running'}
# FUSION -> 5 classes distinctes
KEEP={2:0, 3:1,12:1, 4:2,6:2, 5:3,7:3, 8:4}
CLASS_NAMES=['mange','marche','debout','couché','boit']
CLIP_LEN, SIZE = 16, 112
_BEH=re.compile(r'_beh(\d+)_')

def build_cache():
    print('⏳ décompression (une seule fois)...')
    os.makedirs('/content/cvb',exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall('/content/cvb')
    rf=glob.glob('/content/cvb/**/raw_frames',recursive=True)
    assert rf,'raw_frames introuvable'
    root=os.path.dirname(rf[0])
    X,Y=[],[]
    dirs=sorted(glob.glob(os.path.join(root,'raw_frames','*')))
    for k,d in enumerate(dirs):
        m=_BEH.search(os.path.basename(d));
        if not m: continue
        b=int(m.group(1))
        if b not in KEEP: continue
        fr=sorted(glob.glob(os.path.join(d,'*.jpg')))
        if len(fr)<CLIP_LEN: continue
        idx=np.linspace(0,len(fr)-1,CLIP_LEN).astype(int)
        clip=np.stack([cv2.cvtColor(cv2.resize(cv2.imread(fr[j]),(SIZE,SIZE)),cv2.COLOR_BGR2RGB) for j in idx])
        X.append(clip); Y.append(KEEP[b])
        if k%50==0: print(f'  {k}/{len(dirs)}')
    X=np.stack(X).astype('uint8'); Y=np.array(Y)
    print('💾 sauvegarde du cache dans Drive...')
    np.savez_compressed(CACHE, X=X, y=Y)
    return X,Y

if os.path.exists(CACHE):
    print('✅ Cache trouvé — AUCUNE décompression !')
    d=np.load(CACHE); X,y=d['X'],d['y']
else:
    X,y=build_cache()

print('clips:',len(X),'· classes:',CLASS_NAMES)
print('répartition:',{CLASS_NAMES[k]:v for k,v in sorted(Counter(y.tolist()).items())})

## 4. Entraînement — sampler équilibré + augmentation + scheduler


In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights

MEAN=torch.tensor([0.43216,0.394666,0.37645]).view(3,1,1,1)
STD =torch.tensor([0.22803,0.22145,0.216989]).view(3,1,1,1)

class Clips(Dataset):
    def __init__(self,X,y,augment=False): self.X=X;self.y=y;self.augment=augment
    def __len__(self): return len(self.X)
    def __getitem__(self,i):
        x=torch.from_numpy(self.X[i]).float().div_(255.).permute(3,0,1,2).contiguous()
        if self.augment and np.random.rand()<0.5: x=torch.flip(x,dims=[3])
        return (x-MEAN)/STD, int(self.y[i])

device='cuda'
rng=np.random.default_rng(0); perm=rng.permutation(len(X))
nv=max(1,int(0.15*len(X)))
vi,ti=perm[:nv],perm[nv:]
tr=Clips(X[ti],y[ti],augment=True); va=Clips(X[vi],y[vi])

counts=Counter(y[ti].tolist())
w=[1.0/counts[int(l)] for l in y[ti]]
sampler=WeightedRandomSampler(w,num_samples=len(ti),replacement=True)
tl=DataLoader(tr,batch_size=16,sampler=sampler,num_workers=2,pin_memory=True)
vl=DataLoader(va,batch_size=16,num_workers=2,pin_memory=True)

model=r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)
model.fc=nn.Linear(model.fc.in_features,len(CLASS_NAMES)); model=model.to(device)
opt=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-2)
EPOCHS=20
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
lossf=nn.CrossEntropyLoss()

best=0.0
for ep in range(EPOCHS):
    model.train()
    for xb,yb in tl:
        xb,yb=xb.to(device),yb.to(device)
        opt.zero_grad(); lossf(model(xb),yb).backward(); opt.step()
    sched.step()
    model.eval(); c=t=0; per=Counter(); ok=Counter()
    with torch.no_grad():
        for xb,yb in vl:
            p=model(xb.to(device)).argmax(1).cpu()
            c+=(p==yb).sum().item(); t+=yb.numel()
            for a,b in zip(yb.tolist(),p.tolist()): per[a]+=1; ok[a]+=(a==b)
    acc=c/max(1,t); print(f'epoch {ep+1}/{EPOCHS}  val_acc={acc:.3f}')
    if acc>=best:
        best=acc
        torch.save({'state_dict':model.cpu().state_dict(),'class_names':CLASS_NAMES,
                    'clip_len':CLIP_LEN,'size':SIZE,'mean':MEAN,'std':STD,'arch':'r2plus1d_18'},
                   '/content/drive/MyDrive/behavior_video.pt'); model.to(device)
print('meilleur val_acc =',round(best,3))
print('par classe :',{CLASS_NAMES[k]:round(ok[k]/max(1,per[k]),2) for k in sorted(per)})

## 5. Récupérer le modèle

`behavior_video.pt` (5 classes) est dans ton Drive → copie-le dans
`boeuf-tracker/`. `behavior_video.py` s'adapte tout seul (classes lues
dans le checkpoint). Relance `./dev.sh`.
